In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType,StructField,StringType
for name,default in [('catalog','electrocasa_dev'),('foreign_catalog','electrocasa_sql')]:
    dbutils.widgets.text(name,default)
catalog=dbutils.widgets.get('catalog'); assert catalog in ('electrocasa_dev','electrocasa')
landing=f'/Volumes/{catalog}/bronze/landing'
assert dbutils.fs.ls(f'{landing}/catalogo') and dbutils.fs.ls(f'{landing}/empleados')
# Catálogo de baja frecuencia: snapshot completo, reemplazo atómico de tabla managed.
p=(spark.read.option('multiLine','true').json(f'{landing}/catalogo/*.json')
   .withColumn('_ingested_at',F.current_timestamp())
   .withColumn('_source_file',F.col('_metadata.file_path'))
   .withColumn('_batch_id',F.current_date().cast('string')))
p.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{catalog}.bronze.productos')


In [ ]:
# RR. HH.: COPY INTO evita recargar archivos ya procesados (force=false).
# Definimos columnas como STRING para conservar exactamente el valor de origen.
cols='id_empleado nombre dni email salario sucursal_id cargo tipo_evento fecha_evento'.split()
schema=', '.join(f'`{x}` STRING' for x in cols)
spark.sql(f'CREATE TABLE IF NOT EXISTS `{catalog}`.`bronze`.`empleados` ({schema}, _ingested_at TIMESTAMP, _source_file STRING, _batch_id STRING) USING DELTA')
projection=', '.join(f'`{x}`' for x in cols)
spark.sql(f"""COPY INTO `{catalog}`.`bronze`.`empleados`
 FROM (SELECT {projection}, current_timestamp() AS _ingested_at,
              _metadata.file_path AS _source_file,
              cast(_metadata.file_modification_time AS STRING) AS _batch_id
       FROM '{landing}/empleados/')
 FILEFORMAT = CSV
 FORMAT_OPTIONS ('header'='true', 'inferSchema'='false')
 COPY_OPTIONS ('force'='false')""")


In [ ]:
# Tracking: lectura a través de Lakehouse Federation; el catálogo foráneo
# está configurado con secretos mediante 04_create_connection.ipynb.
foreign_catalog=dbutils.widgets.get('foreign_catalog')
assert foreign_catalog.replace('_','').isalnum()
columns='tracking_id pedido_id courier estado_entrega sucursal_origen fecha_actualizacion'.split()
df=spark.table(f'`{foreign_catalog}`.`dbo`.`TrackingEnvios`').select(*columns)
(df.withColumn('_ingested_at',F.current_timestamp())
   .withColumn('_source_file',F.lit('azure_sql:dbo.TrackingEnvios'))
   .withColumn('_batch_id',F.current_date().cast('string'))
   .write.format('delta').mode('overwrite').option('overwriteSchema','true')
   .saveAsTable(f'{catalog}.bronze.tracking'))
print('Snapshots y COPY INTO finalizados; tracking consultado por federación en Azure SQL')
